# 🎙️ ProXMax — English → Bangla ভিডিও ডাবিং

> 📌 **`Runtime → Run all` চাপলেই কাজ শেষ।** আগে `Runtime → Change runtime type → T4 GPU`
> সেট করে নিন। **টার্মিনালে কিছু টাইপ করতে হবে না** — শুধু **ধাপ ৬**-এর সেলে
> আপনার Gemini API key পেস্ট করবেন। কণ্ঠের জন্য কিছুই লাগে না (বিল্ট-ইন পুরুষ বাংলা ভয়েস)।

এই নোটবুকটি Google Drive-এর `Video/input` ফোল্ডারের **সব English ভিডিও** একে একে বাংলায় ডাবিং করে `Video/output`-এ রাখে।

**এই সংস্করণে নতুন কী:**

- 🎤 **বিল্ট-ইন পুরুষ কণ্ঠ** (`bn-BD-PradeepNeural`) — কোনো নমুনা, token বা অনুমতি লাগে না। **ক্লোনিং সম্পূর্ণ বাদ**।
- 🏷️ **নাম-শব্দের আলাদা ফাইল** (`hotwords.txt`) — ফুটবল/FC Mobile-এর শত শত খেলোয়াড়, ক্লাব ও ইভেন্টের নাম; Whisper এখন ঠিক লিখবে।
- ⚡ **মডেল আগেই নামে** — ধাপ ৫-এ Whisper `large-v3` একবার নামে, তাই ভিডিও প্রসেসিংয়ের সময় আর ডাউনলোডের অপেক্ষা নেই।
- 🔎 **API যাচাই** — key দেওয়ার পরেই একটা ছোট অনুবাদ করে দেখা হয় API সত্যিই কাজ করছে কি না; ৪০০/key সমস্যা আগেই ধরা পড়ে।
- ⏱️ **ফাস্ট ইনস্টল** — ভারী CUDA হুইল এখন দরকার হলেই কেবল ইনস্টল হয় (সাধারণত লাগেই না)।
- ✂️ **নীরবতা-সচেতন chunking, rolling prompt, duration-aware অনুবাদ, কোনো overlap নেই** — আগের মতোই।

### শুরুর আগে কেবল দুটি জিনিস দরকার

| দরকার | কোথায় |
|---|---|
| ভিডিও (mp4/mkv/mov/avi) | `MyDrive/Video/input/` |
| **Gemini API key** (অনুবাদের জন্য) | ধাপ ৬-এর সেলে পেস্ট করবেন (অথবা Colab Secrets-এ `GEMINI_API_KEY`) |

> কণ্ঠের জন্য **কিছুই লাগবে না** — কোনো রেকর্ডিং, Hugging Face token বা লাইসেন্স সম্মতি নয়।

## 📂 ধাপ ১: Google Drive mount

Drive-এ থাকা ভিডিও পড়তে এবং ফলাফল সেভ করতে Drive যুক্ত করা হচ্ছে।

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

INPUT_DIR = '/content/drive/MyDrive/Video/input'
OUTPUT_DIR = '/content/drive/MyDrive/Video/output'

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

## ⬇️ ধাপ ২: রিপোজিটরি clone করা

আগে থেকে clone করা থাকলে সেটি আপডেট (pull) করা হবে — নতুন কোড নিজে থেকেই চলে আসবে।

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/redwan-primeAmax/ProXMax.git'
REPO_DIR = '/content/ProXMax'


def _git(*args):
    return subprocess.run(
        ['git', '-C', REPO_DIR, *args], capture_output=True, text=True, check=False
    )


if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('আগের clone পাওয়া গেছে — সর্বশেষ কোডে আপডেট করা হচ্ছে…')
    _git('fetch', '--depth', '1', 'origin', 'main')
    if _git('merge', '--ff-only', 'origin/main').returncode != 0:
        print('স্থানীয় পরিবর্তন থাকায় fast-forward হলো না — জোর করে সর্বশেষ কোডে আনা হচ্ছে…')
        _git('reset', '--hard', 'origin/main')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

print('বর্তমান commit:', _git('log', '-1', '--oneline').stdout.strip())

## 📦 ধাপ ৩: Dependency ইনস্টল (দ্রুত)

`requirements.txt`-এ শুধু যা সত্যিই লাগে তা-ই আছে — ভারী `nvidia-*` CUDA হুইল
(≈৭০০ MB) বাদ দেওয়া হয়েছে, Colab-এ সাধারণত সেগুলো লাগেই না। দরকার হলে ধাপ ৫
নিজেই ইনস্টল করে নেবে।

> সাধারণত **৩০–৬০ সেকেন্ড**। একই সেশনে দ্বিতীয়বার চালালে প্রায় সঙ্গে সঙ্গে শেষ।

In [ ]:
%cd /content/ProXMax
# --progress-bar off: নোটবুকে অগ্রগতির লাইন কম; PIP_DISABLE...: পুরোনো ভার্সনের জন্য
import os
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
!pip install -q --no-input --progress-bar off -r requirements.txt
print('✅ Dependency ইনস্টল শেষ।')

In [ ]:
import shutil
import subprocess

# ffmpeg না থাকলে ইনস্টল করা (Colab-এ সাধারণত আগে থেকেই থাকে)
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    print('ffmpeg পাওয়া যায়নি — ইনস্টল করা হচ্ছে…')
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)

print('ffmpeg:', shutil.which('ffmpeg'))
print('ffprobe:', shutil.which('ffprobe'))

## ⚙️ ধাপ ৪: দ্রুত সেটিংস (ঐচ্ছিক)

না বদলালেও চলবে — ডিফল্ট মানই সর্বোচ্চ গুণমানের। শুধু দরকার হলে বদলান।

In [ ]:
WHISPER_MODEL = 'large-v3'              # 'large-v3' (best) | 'medium' (faster)
TTS_VOICE = 'bn-BD-PradeepNeural'       # পুরুষ (ডিফল্ট)। নারী কণ্ঠ: 'bn-BD-NabanitaNeural'
# ভিডিওর বিশেষ নাম/শব্দ (কমা দিয়ে) — খালি রাখলেই hotwords.txt ফাইল নিজে থেকে পড়া হয়।
# ফাইলে ৭০০+ ফুটবল নাম আছে; নিজের নাম যোগ করতে চাইলে সেগুলো এখানে লিখুন।
HOTWORDS = ''
FORCE_REPROCESS = True                  # True = আগের আউটপুট থাকলেও নতুন করে ডাবিং

args = []
if WHISPER_MODEL != 'large-v3':
    args += ['--model', WHISPER_MODEL]
# সবসময় স্পষ্টভাবে পাঠানো হয় — পুরোনো config.yaml-এ নারী ভয়েস থাকলেও এটিই চূড়ান্ত
args += ['--voice', TTS_VOICE]
if HOTWORDS.strip():
    args += ['--hotwords', HOTWORDS.strip()]
if FORCE_REPROCESS:
    args += ['--force']                 # skip-existing বন্ধ → আবার প্রসেস হবে

print('অতিরিক্ত আর্গুমেন্ট:', args or '(কিছু নেই — ডিফল্ট)')
print('কণ্ঠ:', TTS_VOICE, '— পুরুষ' if 'Pradeep' in TTS_VOICE else '— নারী')
print('Whisper মডেল:', WHISPER_MODEL)

## 🧠 ধাপ ৫: মডেল আগেই নামানো (স্বয়ংক্রিয়)

Whisper `large-v3`-এর ওজন (≈৩ GB) **এখানেই একবার** নামে। ফলে ধাপ ৭-এ ভিডিও
শুরু করার সময় আর ডাউনলোডের জন্য অপেক্ষা করতে হয় না, আর বারবার ডাউনলোডের
সমস্যাও আর থাকে না (`local_files_only` — ক্যাশে থাকলে নেটওয়ার্ক ছোঁয়াও হয় না)।

এই সেলটি GPU-র CUDA লাইব্রেরিও যাচাই করে; কোনোটা না পেলে তখনই ইনস্টল করে।
কিছু করতে হয় না — শুধু রান করুন।

In [ ]:
%cd /content/ProXMax
import os
WHISPER_MODEL = globals().get('WHISPER_MODEL', 'large-v3')   # ধাপ ৪ না চালালেও ডিফল্ট
%env WHISPER_MODEL={WHISPER_MODEL}
!python main.py --setup-runtime --model $WHISPER_MODEL

## 🔑 ধাপ ৬: Gemini API key + যাচাই (একমাত্র বাধ্যতামূলক ধাপ)

অনুবাদ হয় **Google Gemini API** দিয়ে (ডিফল্ট মডেল `gemini-3.5-flash-lite`)।

- key নিন: https://aistudio.google.com/app/apikey
- নিচের সেলের `GEMINI_API_KEY = ''` এর ভেতরে **পেস্ট করে সেলটি রান করুন**।
- তার ঠিক পরের সেলটি **API সত্যিই কাজ করছে কি না** যাচাই করে (একটা ছোট অনুবাদ)
  — key/মডেল ঠিক না থাকলে এখানেই স্পষ্ট বোঝা যাবে, ১০ মিনিটের রান নষ্ট হবে না।
- Colab Secrets-এ `GEMINI_API_KEY` রাখলে সেটিও নিজে থেকেই পড়া হবে।

In [ ]:
# ⬇️ এখানে আপনার Gemini API key পেস্ট করুন (নাহলে Colab Secrets-এ GEMINI_API_KEY রাখুন)
GEMINI_API_KEY = ''

import os

if not GEMINI_API_KEY.strip():
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
    except Exception:
        GEMINI_API_KEY = (
            os.environ.get('GEMINI_API_KEY', '') or os.environ.get('GOOGLE_API_KEY', '')
        )

GEMINI_API_KEY = GEMINI_API_KEY.strip()

if not GEMINI_API_KEY:
    raise SystemExit(
        '❌ Gemini API key পাওয়া যায়নি — কিছুই শুরু করা হলো না।\n'
        '   উপরের GEMINI_API_KEY = \'\' এর ভেতরে key পেস্ট করে এই সেলটি আবার রান করুন '
        '(অথবা Colab Secrets-এ GEMINI_API_KEY রাখুন)।\n'
        '   key নিন: https://aistudio.google.com/app/apikey'
    )

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print('✅ Gemini key সেট হয়েছে (শেষ ৪ অক্ষর: …%s)' % GEMINI_API_KEY[-4:])

In [ ]:
%cd /content/ProXMax
# API যাচাই: ছোট একটা অনুবাদ করে দেখা হয় — ব্যর্থ হলে এখানেই পরিষ্কার বার্তা
!python main.py --check-gemini

## ▶️ ধাপ ৭: ডাবিং শুরু

`Video/input`-এর সব ভিডিও একটার পর একটা প্রসেস হবে। **নিচের সেলের আউটপুটে লগ
লাইভ আসবে** — কোথায় আছে তা দেখা যাবে, আটকে গেছে ভাবতে হবে না।

প্রথম রানে Whisper মডেল লোড হতে ~১০–২০ সেকেন্ড, তারপর ১০ মিনিটের ভিডিও
সাধারণত **২–৩ মিনিটে** শেষ হয়।

In [ ]:
import os
import queue
import subprocess
import sys
import threading
import time

%cd /content/ProXMax

if not os.environ.get('GEMINI_API_KEY') and not os.environ.get('GOOGLE_API_KEY'):
    raise SystemExit('❌ আগে ধাপ ৬-এর সেলটি রান করুন (Gemini API key)।')

args = globals().get('args', [])          # ধাপ ৭ না চালালেও চলবে
command = [sys.executable, '-u', 'main.py'] + list(args)

print('চালানো হচ্ছে:', ' '.join(command), flush=True)
print('নিচে লগ লাইভ আসবে… (মডেল ধাপ ৫-এ নামানো হয়ে গেছে, তাই দ্রুত শুরু হবে)\n', flush=True)

env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
proc = subprocess.Popen(
    command,
    cwd='/content/ProXMax',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
    env=env,
)

lines: 'queue.Queue' = queue.Queue()


def _pump() -> None:
    assert proc.stdout is not None
    for line in proc.stdout:
        lines.put(line)
    lines.put(None)


threading.Thread(target=_pump, daemon=True).start()

printed = 0
last_output = time.monotonic()
while True:
    try:
        line = lines.get(timeout=1.0)
    except queue.Empty:
        # ৬০ সেকেন্ড চুপ থাকলে বোঝানো হয় কাজ এগোচ্ছে — আটকে যায়নি
        if time.monotonic() - last_output > 60:
            print('… এখনো কাজ চলছে (নতুন লগ নেই; Whisper মডেল ডাউনলোড/লোড হতে পারে)।',
                  flush=True)
            last_output = time.monotonic()
        continue
    if line is None:
        break
    print(line, end='', flush=True)
    printed += 1
    last_output = time.monotonic()

exit_code = proc.wait()
print('\n— শেষ: exit code %d —' % exit_code, flush=True)

if printed == 0:
    print('⚠️ কোনো লগ আসেনি। ধাপ ৪ (pip install) রান করেছেন কি না দেখুন, '
          'তারপর এই সেলটি আবার চালান।', flush=True)
elif exit_code != 0:
    print('ℹ️ উপরের লগে ❌ চিহ্নিত লাইনটিই আসল কারণ।', flush=True)

## 🎉 ধাপ ৮: ফলাফল দেখা

প্রতিটি ভিডিওর নামের শেষে `_bn.mp4` — সেটাই ডাবিং করা ভিডিও। পাশে `_en.srt` (মূল) ও `_bn.srt` (বাংলা) সাবটাইটেল।

In [ ]:
import os
import subprocess

files = sorted(os.listdir(OUTPUT_DIR)) if os.path.isdir(OUTPUT_DIR) else []
videos = [f for f in files if f.lower().endswith('.mp4')]

print(f'মোট ফাইল: {len(files)} | ডাবিং করা ভিডিও: {len(videos)}\n')
for name in files:
    path = os.path.join(OUTPUT_DIR, name)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  {name:45s} {size_mb:7.1f} MB')

if videos:
    print('\nDrive থেকে সরাসরি দেখা/ডাউনলোড করা যাবে:', os.path.join(OUTPUT_DIR, videos[0]))

In [ ]:
# প্রথম ডাবিং করা ভিডিওটি ব্রাউজারে চালিয়ে শোনা (Drive থেকে)
from IPython.display import HTML, display
import os

videos = [f for f in sorted(os.listdir(OUTPUT_DIR)) if f.lower().endswith('_bn.mp4')]
if videos:
    path = os.path.join(OUTPUT_DIR, videos[0])
    display(HTML(f'<video controls width="640" src="/content/drive/MyDrive/Video/output/{videos[0]}"></video>'))
    print('ভিডিও:', path)
else:
    print('কোনো ডাবিং করা ভিডিও পাওয়া যায়নি — ধাপ ৭-এর লগ দেখুন।')

## 🧹 ধাপ ৯: GPU মেমরি ছেড়ে দেওয়া

ডাবিং শেষে প্রোগ্রাম নিজেই মডেল আনলোড করে GPU ছেড়ে দেয়। তবু নিশ্চিত করতে এই সেলটি রান করুন।

In [ ]:
import gc

for name in ('pipeline', 'model', 'tts_engine'):
    globals()[name] = None
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        used = torch.cuda.memory_allocated() / (1024 ** 3)
        print(f'GPU মেমরি এখন ব্যবহৃত: {used:.2f} GB (খালি করা হয়েছে)')
    else:
        print('GPU পাওয়া যায়নি — CPU মোড।')
except Exception as exc:
    print('torch ছাড়াই পরিষ্কার করা হলো:', exc)

## 🩺 সমস্যা হলে

| লক্ষণ | কারণ ও সমাধান |
|---|---|
| `⏭️ আগের আউটপুট পাওয়া গেছে … বাদ দেওয়া হলো` | আগের রানের `video_bn.mp4` এখনো আছে। ধাপ ৪-এ `FORCE_REPROCESS = True` রাখুন (ডিফল্ট) |
| ধাপ ৭-এ কিছুই দেখা যাচ্ছে না | লগ লাইভ আসে। তবু কিছু না এলে ধাপ ৩ (`pip install -r requirements.txt`) রান করে আবার চেষ্টা করুন |
| কোড পুরোনো (নতুন ফিচার নেই) | ধাপ ২-এর সেল আবার রান করুন — সর্বশেষ `main` commit এনে দেখায় |
| `Gemini API key পাওয়া যায়নি` | ধাপ ৬-এর `GEMINI_API_KEY = ''`-এর ভেতরে key পেস্ট করে সেলটি রান করুন |
| **`❌ Gemini API কাজ করছে না` / `400`** | ধাপ ৬-এর যাচাই-সেলে স্পষ্ট কারণ দেখাবে। key নতুন করে পেস্ট করুন; কোড নিজেই ৪০০ পেলে schema → thinking → system_instruction পরপর বাদ দিয়ে আবার চেষ্টা করে (কোনো অপেক্ষা নেই) |
| `edge-tts` ব্যর্থ / কণ্ঠ আসছে না | ইন্টারনেট সংযোগ দেখুন; `TTS_VOICE` বৈধ `bn-BD-*` কি না দেখুন |
| অডিও extract-এ আটকে যায় | লাইভ শতাংশ দেখাবে; নির্দিষ্ট সময় অগ্রগতি না হলে থেমে software decoding-এ নামবে বা স্পষ্ট error দেবে |
| `CUDA out of memory` | Runtime → Restart, ধাপ ৭ আবার চালান; কোড নিজেই TTS-এর আগে Whisper আনলোড করে |
| `Gemini API error (429)` | quota শেষ — কিছুক্ষণ অপেক্ষা করুন; কোড নিজেই ৭ বার retry করে, প্রতিবার **সর্বোচ্চ ২ সেকেন্ড** |
| কণ্ঠ মেয়েল শোনায় | ধাপ ৭-এর লগে `কণ্ঠ (বিল্ট-ইন পথ): bn-BD-PradeepNeural | লিঙ্গ: male` দেখুন। না থাকলে ধাপ ২ (নতুন কোড) ও ধাপ ৪ রান করুন |
| অনুবাদে শব্দ ভুল (নাম, গেম-টার্ম) | `hotwords.txt`-এ নিজের নাম যোগ করুন (অথবা ধাপ ৪-এর `HOTWORDS`) — Whisper এখনই এগুলো ঠিকভাবে লেখে |
| মাঝপথে থেমে থেমে কথা শোনায় | এখন প্রতিটি **পুরো বাক্য** এক clip-এ পড়া হয় |
| এক লাইন বলে অনেকক্ষণ ফাঁক, তারপর বাকিটা | TTS-এর দুই প্রান্তের নীরবতা ছাঁটা হয়, ছোট ক্লিপ স্লট ভরাতে সামান্য ধীর হয়, বড় ফাঁকের লাইন Gemini দিয়ে ভরাট করে আবার TTS হয়। লগে `🕳️ … লাইন ভরাট` ও `নীরবতা X s (Y%)` দেখুন |
| কথাগুলো তাড়াহুড়ো শোনায় | শুরুতেই কণ্ঠের প্রকৃত গতি মাপা হয়; লগে `🎚️ কণ্ঠের প্রকৃত গতি` দেখুন |
| ডাবিং ধীর লাগছে | `WHISPER_MODEL='medium'` করুন |
| মডেল আবার ডাউনলোড হচ্ছে | ধাপ ৫ রান করুন — মডেল আগেই নামানো থাকবে; লগে `ওজন আগেই নামানো আছে` দেখবেন |

বিস্তারিত: রিপোর `README.md`।